# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kab-s/flyrank/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [17]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    # find the repo root from wherever this kernel started
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

print("Working dir:", os.getcwd())
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found — are you at the repo root?"
print("Starter data found. You're ready.")

Working dir: /content/flyrank-ml-internship-starter/flyrank-ml-internship-starter
Starter data found. You're ready.


## 1. My lane (or freestyle) and why

**Lane chosen: Refresh / Content Opportunity Scoring (Lane 2)**

I'm choosing the Refresh/Content Opportunity Scoring lane because it is observed that a substantial portion of the content portfolio requires attention for refresh, expansion, or optimization:

*   **13,191 pages (44% of the dataset)** meet at least one refresh criteria
*   These pages represent **51.2% of all search impressions (79.9M impressions)**
*   **16,262 pages (54.2%)** are actively declining in visibility

Given editors' limited capacity for manual review, comprehensively addressing these opportunities would require significant workforce investment over an extended period, making manual prioritization infeasible at scale.

The central question is: **Which pages should an editor prioritize for refresh, expansion, or optimization?** A transparent baseline rule can start the ranking, but machine learning can discover patterns across multiple signals (freshness, position, CTR, engagement, content depth) that a simple rule would miss. This lane directly supports real editorial decisions with measurable business impact.

In [18]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Load data and verify the opportunity scope
import pandas as pd
import numpy as np

df = pd.read_csv('data/raw/content_refresh_anonymized.csv')

print(f"Total pages: {len(df):,}")
print(f"Declining pages: {(df['trend_direction'].str.lower() == 'down').sum():,} ({(df['trend_direction'].str.lower() == 'down').sum()/len(df)*100:.1f}%)")

# Count refresh candidates (any of: declining + demand, stale + visible, thin + visible)
refresh_candidates = df[
    ((df['trend_direction'].str.lower() == 'down') & (df['impressions_90d'] >= 100)) |
    ((df['days_since_last_update'] >= 180) & (df['impressions_90d'] >= 500)) |
    ((df['word_count'] > 0) & (df['word_count'] < 1200) & (df['impressions_90d'] >= 250))
]
print(f"\nPages meeting ANY refresh criteria: {len(refresh_candidates):,} ({len(refresh_candidates)/len(df)*100:.1f}%)")
print(f"Total impressions on these pages: {refresh_candidates['impressions_90d'].sum():,.0f}")
print(f"Percentage of all impressions: {refresh_candidates['impressions_90d'].sum()/df['impressions_90d'].sum()*100:.1f}%")

Total pages: 30,000
Declining pages: 16,262 (54.2%)

Pages meeting ANY refresh criteria: 13,191 (44.0%)
Total impressions on these pages: 79,950,770
Percentage of all impressions: 51.2%


## 2. The question: decision, action, cost of a wrong call

**The decision:** Which existing content pages should a content editor prioritize for refresh, expansion, or optimization?

**Who acts:** Content editors and SEO managers who have limited capacity to review and update pages.

**The action:** The editor reviews the top-ranked pages and decides whether to:
- Refresh stale content with updated information
- Expand thin pages with more depth
- Optimize titles/meta descriptions for low-CTR pages in good positions
- Protect high-value pages showing early decline signals
- Monitor pages that don't need immediate action

**Cost of a wrong call:**
- **False positive (prioritize wrong page):** Editor time wasted on a page that wouldn't have improved or didn't need urgent attention. Opportunity cost of ~1-2 hours per page.
- **False negative (miss the right page):** A high-value page continues declining. For pages in top 10 positions with 1000+ impressions/month, missing the decline window means continued traffic loss (potentially hundreds of clicks/month). The data shows **3,666 pages in top 10 that are declining** — these are the expensive misses.
- **Ranking error matters more than binary classification:** Getting position 1 vs position 20 wrong is more costly than a binary yes/no error. An editor reviewing the top 50 needs THE RIGHT 50 in the right order.

In [19]:
# Quantify the cost of wrong prioritization

# High-value pages at risk (top 10 position + declining)
high_value_at_risk = df[
    (df['avg_position'] > 0) &
    (df['avg_position'] <= 10) &
    (df['trend_direction'].str.lower() == 'down')
]

print("HIGH-VALUE PAGES AT RISK:")
print(f"Pages in top 10 positions that are declining: {len(high_value_at_risk):,}")
print(f"Total impressions: {high_value_at_risk['impressions_90d'].sum():,.0f}")
print(f"Average position: {high_value_at_risk['avg_position'].mean():.2f}")
print(f"Average impressions per page: {high_value_at_risk['impressions_90d'].mean():.0f}")

print("\nEDITOR CAPACITY CONSTRAINT:")
print(f"If editor can review 50 pages/month: {len(refresh_candidates)/50:.1f} months to review all candidates")
print(f"Need for prioritization: CRITICAL")
print(f"Metric that matters: Precision@50 (what % of top 50 are actually high-value opportunities)")


HIGH-VALUE PAGES AT RISK:
Pages in top 10 positions that are declining: 7,311
Total impressions: 44,876,369
Average position: 6.29
Average impressions per page: 6138

EDITOR CAPACITY CONSTRAINT:
If editor can review 50 pages/month: 263.8 months to review all candidates
Need for prioritization: CRITICAL
Metric that matters: Precision@50 (what % of top 50 are actually high-value opportunities)


## 3. Quick look at the data (2-3 real numbers)

Three numbers that make this lane compelling:

1. **13,152 declining pages with measurable demand** (>= 100 impressions/90d) — these aren't zero-traffic pages; they're pages losing visibility that previously had it

2. **9,759 pages with low CTR (<0.5%) despite good visibility** (500+ impressions, positions 1-20) — if CTR improved to even 1%, that's ~910,000 additional clicks across the dataset

3. **7,076 pages in top 10 positions that are 180+ days old** — these are ranking well NOW but aging. Average position 6.2, representing 57M impressions. These are protection candidates before they start declining.

The opportunity is real, measurable, and too large to tackle without smart prioritization.

In [20]:
# Show the three compelling numbers

# 1. Declining pages with demand
declining_with_demand = df[
    (df['trend_direction'].str.lower() == 'down') &
    (df['impressions_90d'] >= 100)
]
print("1. DECLINING PAGES WITH DEMAND")
print(f"   Count: {len(declining_with_demand):,}")
print(f"   Total impressions at risk: {declining_with_demand['impressions_90d'].sum():,.0f}")
print(f"   Median impressions: {declining_with_demand['impressions_90d'].median():.0f}")

# 2. Low CTR opportunity
low_ctr = df[
    (df['impressions_90d'] >= 500) &
    (df['avg_position'] > 0) &
    (df['avg_position'] <= 20) &
    (df['ctr'] < 0.5)
]
print(f"\n2. LOW CTR OPPORTUNITY")
print(f"   Count: {len(low_ctr):,}")
print(f"   Average CTR: {low_ctr['ctr'].mean():.3f}%")
print(f"   Average position: {low_ctr['avg_position'].mean():.2f}")
print(f"   Potential clicks if CTR improved to 1%: {(low_ctr['impressions_90d'].sum() * 0.01):,.0f}")

# 3. Page 1 aging risk
page_one_aging = df[
    (df['avg_position'] > 0) &
    (df['avg_position'] <= 10) &
    (df['content_age_days'] >= 180)
]
print(f"\n3. PAGE 1 AGING RISK (protection candidates)")
print(f"   Count: {len(page_one_aging):,}")
print(f"   Average position: {page_one_aging['avg_position'].mean():.2f}")
print(f"   Total impressions: {page_one_aging['impressions_90d'].sum():,.0f}")
print(f"   Average age: {page_one_aging['content_age_days'].mean():.0f} days")

1. DECLINING PAGES WITH DEMAND
   Count: 13,152
   Total impressions at risk: 79,887,612
   Median impressions: 1620

2. LOW CTR OPPORTUNITY
   Count: 9,759
   Average CTR: 0.185%
   Average position: 9.48
   Potential clicks if CTR improved to 1%: 909,680

3. PAGE 1 AGING RISK (protection candidates)
   Count: 7,076
   Average position: 6.21
   Total impressions: 57,187,548
   Average age: 342 days


## 4. Careful words: what I can and can't claim

**What I CAN claim:**

- **Observed:** "It is observed that 54% of pages show declining trend_direction, and pages with X characteristics are more likely to show declining patterns"
- **Directional:** "Pages with X characterstics (longer staleness, thinner content, and lower CTR) are associated with declining visibility"
- **Decision-support:** "The model produces a ranked priority queue to help editors decide which pages to review first, measured by Precision@50"
- **Comparative:** "The learned model ranks pages with 3x better precision at the top 50 compared to a transparent baseline rule"

**What I CANNOT claim:**

- ❌ "This predicts Google's ranking algorithm" — I'm modeling observed patterns in traffic data, not reverse-engineering Google
- ❌ "Refreshing a page CAUSES traffic recovery" — I have no experimental data, only observational. Causality requires an A/B test or intervention study I don't have
- ❌ "This model guarantees results" — It's decision-support, not automation. Human editorial judgment is still required

In [21]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.